# 🧭 CDM Compass
**Clinical Data Management Knowledge Navigator**

Ask questions about your CDM/CDS documents in plain English and get answers with inline citations.
Documents are classified by folder — regulatory requirements and opinion papers are treated differently in every answer.

### 📋 How to use this notebook
Run each block **from top to bottom**, in order. You only need to re-run from Block 3 onwards after adding new documents.

| Block | What it does | Run again? |
|-------|-------------|------------|
| 1 | Install packages | Only once |
| 2 | Import libraries | Each session |
| 3 | Load documents | When you add files |
| 4 | Chunk text | When you add files |
| 5 | Build search index | When you add files |
| 6 | Connect to Ollama LLM | Each session |
| 7 | Ask a question | Anytime! |
| 8 | Interactive Q&A loop | Anytime! |

### 📁 Document authority
```
data/raw/
├── regulatory/    ← binding requirements (ICH, FDA, EMA)
└── opinion/       ← position papers, white papers (SCDM, JSCDM)
```
Files placed anywhere else will raise an error — nothing is ever classified by accident.


## 📦 Block 1 — Install Required Packages

Run this once. It installs everything CDM Compass needs.

> ⚠️ **Before running Block 6**, install Ollama separately:
> 1. Go to [https://ollama.com](https://ollama.com) and download the app
> 2. Open Terminal and run: `ollama pull mistral` (or `ollama pull qwen2.5:14b` for alternative model)
> 3. Ollama runs in the background automatically


In [1]:
# Install all required packages — run this once
import sys

!{sys.executable} -m pip install --upgrade pip --quiet
!{sys.executable} -m pip install "sentence-transformers[cross-encoder]" chromadb pypdf openpyxl python-docx python-pptx rank-bm25 requests tqdm jupyter ipywidgets --quiet
!{sys.executable} -m pip install --upgrade jupyter ipywidgets --quiet

print("\u2705 All packages installed successfully!")


✅ All packages installed successfully!


## 📚 Block 2 — Import Libraries

Loads all tools into memory. Run this at the start of every session.

All imports live here — no other block adds imports.


In [2]:
import os
import re
import json
import csv
import textwrap
import xml.etree.ElementTree as ET
import requests
import numpy as np
import torch
import chromadb
from rank_bm25 import BM25Okapi

from pypdf import PdfReader
from pptx import Presentation
from sentence_transformers import SentenceTransformer, CrossEncoder
from tqdm.notebook import tqdm
import openpyxl
import docx  # python-docx

print("\u2705 All libraries imported successfully!")


✅ All libraries imported successfully!


## 📁 Block 3 — Load Documents

**👇 Only change this line** if your documents folder is in a different location.

Documents must be placed in one of two subfolders — this determines how they are treated:

| Subfolder | Authority | Use for |
|-----------|-----------|---------|
| `regulatory/` | ⚖️ Regulatory | ICH, FDA, EMA — binding requirements |
| `opinion/` | 💬 Opinion | SCDM, JSCDM, white papers, position papers |

**Supported file types:** PDF, PPTX, XLSX, DOCX, JSON, XML, CSV, TXT

> ⚠️ Old-format `.ppt` files are not supported. Open in PowerPoint → File → Save As → .pptx first.


In [3]:
# ============================================================
# Settings
# ============================================================
DOC_FOLDER = "../data/raw/"  # ← change if your folder is elsewhere
# ============================================================

def load_pdf(path):
    reader = PdfReader(path)
    sections = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            sections.append((f"page {i}", text.strip()))
    return sections

def load_pptx(path):
    prs = Presentation(path)
    sections = []
    for i, slide in enumerate(prs.slides, start=1):
        texts = []
        for shape in slide.shapes:
            if hasattr(shape, "text") and shape.text.strip():
                texts.append(shape.text.strip())
        if slide.has_notes_slide and slide.notes_slide.notes_text_frame:
            notes = slide.notes_slide.notes_text_frame.text.strip()
            if notes:
                texts.append(f"[Speaker Notes: {notes}]")
        text = "\n".join(texts)
        if text.strip():
            sections.append((f"slide {i}", text.strip()))
    return sections

def load_xlsx(path):
    wb = openpyxl.load_workbook(path, data_only=True)
    sections = []
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        rows = []
        for row in ws.iter_rows(values_only=True):
            row_str = "\t".join(str(c) if c is not None else "" for c in row)
            if row_str.strip():
                rows.append(row_str)
        text = "\n".join(rows)
        if text.strip():
            sections.append((sheet_name, text.strip()))
    return sections

def load_docx(path):
    doc = docx.Document(path)
    text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
    if text.strip():
        return [("document", text.strip())]
    return []

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    text = json.dumps(data, indent=2)
    return [("document", text)]

def load_xml(path):
    tree = ET.parse(path)
    root = tree.getroot()
    text = " ".join(elem.text for elem in root.iter() if elem.text and elem.text.strip())
    if text.strip():
        return [("document", text.strip())]
    return []

def load_csv(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        for row in reader:
            rows.append("\t".join(row))
    text = "\n".join(rows)
    return [("document", text)]

def load_txt(path):
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    if text.strip():
        return [("document", text.strip())]
    return []

def classify_authority(filepath):
    rel_path = os.path.relpath(filepath, DOC_FOLDER).replace("\\", "/")
    if rel_path.startswith("regulatory/"):
        return "regulatory"
    elif rel_path.startswith("opinion/"):
        return "opinion"
    else:
        raise ValueError(
            f"Cannot classify: {os.path.basename(filepath)}\n"
            f"   Files must be in 'regulatory/' or 'opinion/' subfolder.\n"
            f"   Detected path: {rel_path}"
        )

# Main loading logic
documents = []

print(f"\U0001f50d Scanning folder: {DOC_FOLDER}\n")

for root, _, files in os.walk(DOC_FOLDER):
    for fname in sorted(files):
        if fname.startswith(".") or fname.startswith("~"):
            continue

        fpath = os.path.join(root, fname)
        ext = os.path.splitext(fname)[1].lower()

        if ext == ".ppt":
            print(f"  \u26a0\ufe0f  Skipped (old format): {fname}")
            print("     \u2192 Open in PowerPoint \u2192 File \u2192 Save As \u2192 .pptx\n")
            continue

        loaders = {
            ".pdf": load_pdf,
            ".pptx": load_pptx,
            ".xlsx": load_xlsx,
            ".xls": load_xlsx,
            ".docx": load_docx,
            ".json": load_json,
            ".xml": load_xml,
            ".csv": load_csv,
            ".txt": load_txt,
        }

        loader = loaders.get(ext)
        if not loader:
            continue

        try:
            authority = classify_authority(fpath)
            sections = loader(fpath)

            print(f"  \U0001f4c4 Loading: {fname}")
            print(f"     \u2192 {len(sections)} section(s)  [{authority.upper()}]")

            for sect_id, text in sections:
                documents.append({
                    "source": fpath,
                    "page": sect_id,
                    "text": text,
                    "authority": authority,
                })

        except Exception as e:
            print(f"  \u274c Error loading {fname}: {e}")

print(f"\n\u2705 Loaded {len(documents)} section(s) from {len(set(d['source'] for d in documents))} document(s).")


🔍 Scanning folder: ../data/raw/

  ❌ Error loading LOINC Public Review Comments.xlsx: Cannot classify: LOINC Public Review Comments.xlsx
   Files must be in 'regulatory/' or 'opinion/' subfolder.
   Detected path: OPINION/LOINC/LOINC Public Review Comments.xlsx
  ❌ Error loading LOINC to LB Mapping-Read Me.pdf: Cannot classify: LOINC to LB Mapping-Read Me.pdf
   Files must be in 'regulatory/' or 'opinion/' subfolder.
   Detected path: OPINION/LOINC/LOINC to LB Mapping-Read Me.pdf
  ❌ Error loading LOINC_to_LB_Mapping Document_FINAL.csv: Cannot classify: LOINC_to_LB_Mapping Document_FINAL.csv
   Files must be in 'regulatory/' or 'opinion/' subfolder.
   Detected path: OPINION/LOINC/LOINC_to_LB_Mapping Document_FINAL.csv
  ❌ Error loading LOINC_to_LB_Mapping Document_FINAL.xlsx: Cannot classify: LOINC_to_LB_Mapping Document_FINAL.xlsx
   Files must be in 'regulatory/' or 'opinion/' subfolder.
   Detected path: OPINION/LOINC/LOINC_to_LB_Mapping Document_FINAL.xlsx
  ❌ Error loading Guidan

## ✂️ Block 4 — Chunk Documents

Breaks each section into smaller chunks for more precise retrieval.

**Tuning tip:** Lower `CHUNK_SIZE` (e.g. 200) if you want shorter, more precise citations.


In [4]:
# ============================================================
# Settings
# ============================================================
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50
# ============================================================

def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i : i + size])
        chunks.append(chunk)
        i += size - overlap
    return chunks

chunked = []

print("\u2702\ufe0f Chunking documents...\n")

for doc in tqdm(documents, desc="Chunking"):
    chunks = chunk_text(doc["text"])
    for chunk in chunks:
        chunked.append({
            "source": doc["source"],
            "page": doc["page"],
            "authority": doc["authority"],
            "text": chunk,
        })

print(f"\n\u2705 Created {len(chunked)} chunk(s).")


✂️ Chunking documents...



Chunking: 0it [00:00, ?it/s]


✅ Created 0 chunk(s).


## 🔍 Block 5 — Build Search Index

Creates two indexes:
- **ChromaDB** (semantic / vector search)
- **BM25** (keyword search)

This can take a few minutes on first run — models download once and are cached.


In [5]:
# ============================================================
# Settings
# ============================================================
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
BATCH_SIZE = 100
# ============================================================

print("\U0001f50d Building search index...\n")

# Load embedding and reranking models
print(f"Loading embedding model: {EMBED_MODEL_NAME}...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

print(f"Loading reranking model: {RERANK_MODEL_NAME}...")
rerank_model = CrossEncoder(RERANK_MODEL_NAME)

# ChromaDB
print("\nInitializing ChromaDB...")
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("cdm_compass")
except:
    pass
collection = chroma_client.create_collection("cdm_compass")

print(f"Embedding {len(chunked)} chunk(s) in batches of {BATCH_SIZE}...\n")

for i in tqdm(range(0, len(chunked), BATCH_SIZE), desc="Indexing"):
    batch = chunked[i : i + BATCH_SIZE]
    texts = [c["text"] for c in batch]
    embeddings = embed_model.encode(texts, convert_to_numpy=True).tolist()

    collection.add(
        ids=[str(i + j) for j in range(len(batch))],
        embeddings=embeddings,
        metadatas=[
            {
                "source": c["source"],
                "page": str(c["page"]),
                "authority": c["authority"],
            }
            for c in batch
        ],
        documents=texts,
    )

# BM25
print("\nBuilding BM25 index...")
tokenized = [c["text"].lower().split() for c in chunked]
bm25 = BM25Okapi(tokenized)

print("\n\u2705 Search index ready!")


🔍 Building search index...

Loading embedding model: all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading reranking model: cross-encoder/ms-marco-MiniLM-L-6-v2...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Initializing ChromaDB...
Embedding 0 chunk(s) in batches of 100...



Indexing: 0it [00:00, ?it/s]


Building BM25 index...


ZeroDivisionError: division by zero

## 🤖 Block 6 — Connect to Ollama LLM

Tests connection to your local Ollama server.

**Model options:**
- `mistral` — balanced performance, recommended
- `qwen2.5:14b` — strong alternative with good citation accuracy
- `llama3.2` — another solid choice
- `phi3` — fastest, but less reliable with citation format

To switch models:
1. Change `OLLAMA_MODEL` below
2. In Terminal, run: `ollama pull <model_name>`


In [ ]:
# ============================================================
# Settings
# ============================================================
OLLAMA_MODEL = "mistral"           # Options: "mistral", "qwen2.5:14b", "llama3.2", "phi3"
OLLAMA_URL = "http://localhost:11434/api/generate"
# ============================================================

def ask_ollama(prompt, model=OLLAMA_MODEL, temperature=0.3):
    """Send a prompt to Ollama and return structured result."""
    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": model,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": temperature},
            },
            timeout=120,
        )

        # Raise HTTP errors (e.g. 404)
        response.raise_for_status()

        return {
            "ok": True,
            "response": response.json().get("response", "").strip()
        }

    except requests.exceptions.ConnectionError:
        return {
            "ok": False,
            "error": "connection",
            "message": (
                "Cannot connect to Ollama.\n"
                "→ Make sure Ollama is running (ollama serve or open the Ollama app)"
            )
        }

    except requests.exceptions.HTTPError as e:
        status_code = e.response.status_code

        if status_code == 404:
            return {
                "ok": False,
                "error": "model_not_found",
                "message": (
                    f"Model '{model}' not found.\n"
                    f"→ Run: ollama pull {model}"
                )
            }

        return {
            "ok": False,
            "error": "http_error",
            "message": f"HTTP error {status_code}: {e}"
        }

    except Exception as e:
        return {
            "ok": False,
            "error": "unknown",
            "message": f"Unexpected error: {e}"
        }


# ============================================================
# Connection test
# ============================================================

print(f"\U0001f517 Testing connection to Ollama (model: {OLLAMA_MODEL})...\n")

result = ask_ollama("Respond ONLY with: OK")

if not result["ok"]:
    print(f"\u274c {result['message']}")
elif result["response"].strip().upper() == "OK":
    print(f"\u2705 Ollama is connected and ready!")
    print(f"   Model: {OLLAMA_MODEL}")
else:
    print(f"\u26a0\ufe0f Ollama responded, but output was unexpected.")
    print(f"   Response: {result['response']}")
    print(f"   This usually means the model is working but may not follow instructions precisely.")


## 💬 Block 7 — Ask a Question

Type your question below and run this block.

### ⚙️ Tuning parameters

| Setting | Default | Effect |
|---------|---------|--------|
| `TOP_K` | 10 | Candidates retrieved before reranking — increase if answers miss known sources |
| `FINAL_K` | 5 | Chunks sent to LLM — increase for broader answers, decrease for precision |
| `SEMANTIC_WEIGHT` | 0.7 | Raise for conceptual questions |
| `KEYWORD_WEIGHT` | 0.3 | Raise for specific terms, IDs, clause numbers |


In [ ]:
# ============================================================
# Settings
# ============================================================
MY_QUESTION = "What skills does an experienced Clinical Data Manager of 25+ years need to transition to Data Science?"

TOP_K = 10              # candidates retrieved before reranking
FINAL_K = 5             # chunks sent to LLM after reranking
SEMANTIC_WEIGHT = 0.7   # weight for semantic (vector) search
KEYWORD_WEIGHT = 0.3    # weight for keyword (BM25) search
# ============================================================

# ANSI color codes
def bold(s): return f"\033[1m{s}\033[0m"
def dim(s): return f"\033[2m{s}\033[0m"
def green(s): return f"\033[32;1m{s}\033[0m"
def yellow(s): return f"\033[33;1m{s}\033[0m"
def red(s): return f"\033[31;1m{s}\033[0m"
def blue(s): return f"\033[34;1m{s}\033[0m"
def magenta(s): return f"\033[35;1m{s}\033[0m"

def compress_chunk(chunk_text, question, top_n=3):
    """Extract the most relevant sentences from a chunk."""
    sentences = re.split(r'(?<=[.!?])\s+', chunk_text)
    if len(sentences) <= top_n:
        return chunk_text

    q_lower = question.lower()
    scored = []
    for s in sentences:
        s_lower = s.lower()
        overlap = sum(1 for word in q_lower.split() if word in s_lower)
        scored.append((overlap, s))

    scored.sort(reverse=True, key=lambda x: x[0])
    return " ".join(s for _, s in scored[:top_n])

def ask_question(question, top_k=TOP_K, final_k=FINAL_K):
    print()
    print(bold(f"\U0001f9ed CDM Compass  --  \"{question}\""))
    print(dim(f"   Hybrid search: {int(SEMANTIC_WEIGHT * 100)}% semantic + {int(KEYWORD_WEIGHT * 100)}% keyword  |  reranking top {final_k} of {top_k}"))

    # Semantic search
    q_emb = embed_model.encode([question], convert_to_numpy=True)[0].tolist()
    sem_results = collection.query(query_embeddings=[q_emb], n_results=top_k)

    sem_ids = sem_results["ids"][0]
    sem_dists = sem_results["distances"][0]

    # Keyword search
    q_tokens = question.lower().split()
    kw_scores = bm25.get_scores(q_tokens)
    top_kw_idx = np.argsort(kw_scores)[::-1][:top_k]

    # Hybrid fusion
    hybrid = {}
    for idx, dist in zip(sem_ids, sem_dists):
        idx = int(idx)
        hybrid[idx] = {"sem_score": dist, "kw_score": 0.0}

    kw_max = kw_scores.max() if kw_scores.max() > 0 else 1.0
    for idx in top_kw_idx:
        norm_score = kw_scores[idx] / kw_max
        if idx in hybrid:
            hybrid[idx]["kw_score"] = norm_score
        else:
            hybrid[idx] = {"sem_score": 1.0, "kw_score": norm_score}

    for idx in hybrid:
        hybrid[idx]["hybrid_score"] = (
            SEMANTIC_WEIGHT * hybrid[idx]["sem_score"] +
            KEYWORD_WEIGHT * (1 - hybrid[idx]["kw_score"])
        )

    # Reranking
    rerank_input = []
    for idx in hybrid:
        rerank_input.append({
            "idx": idx,
            "text": chunked[idx]["text"],
            "source": chunked[idx]["source"],
            "page": chunked[idx]["page"],
            "authority": chunked[idx]["authority"],
            "sem_score": hybrid[idx]["sem_score"],
            "kw_score": hybrid[idx]["kw_score"],
            "hybrid_score": hybrid[idx]["hybrid_score"],
        })

    pairs = [[question, r["text"]] for r in rerank_input]
    rerank_scores = rerank_model.predict(pairs)

    for i, r in enumerate(rerank_input):
        r["rerank_score"] = float(rerank_scores[i])

    rerank_input.sort(key=lambda x: x["rerank_score"], reverse=True)
    top_results = rerank_input[:final_k]

    # Sort: regulatory first, then by hybrid score
    top_results.sort(key=lambda x: (
        0 if x["authority"] == "regulatory" else 1,
        x["hybrid_score"]
    ))

    print(dim(f"   {len(top_results)} chunks after reranking, regulatory sources first"))
    print(dim(f"   Asking {OLLAMA_MODEL}..."))

    # Build context with compressed chunks
    context_parts = []
    for i, r in enumerate(top_results, start=1):
        compressed = compress_chunk(r["text"], question, top_n=3)
        fname = os.path.basename(r["source"])
        authority = r["authority"]
        context_parts.append(
            f"[{i}] {fname} (page {r['page']}, {authority}):\n{compressed}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""You are a Clinical Data Management expert assistant.

Answer the user's question using ONLY the provided context. Follow these rules strictly:

1. CITATION FORMAT:
   - Use ONLY bracket numbers: [1], [2], [3], etc.
   - NEVER write filenames, page numbers, or source details inline
   - Citations appear in a separate "Sources" section below your answer

2. LANGUAGE BY SOURCE TYPE:
   - Regulatory sources (ICH, FDA, EMA): use "must", "shall", "is required", "mandates"
   - Opinion sources (SCDM, JSCDM, white papers): use "recommends", "suggests", "proposes", "advises"

3. ANSWER STRUCTURE:
   - Start with a direct answer
   - Support each point with inline citations
   - Keep language clear and professional
   - If context doesn't answer the question, say so

CONTEXT:
{context}

QUESTION: {question}

Provide your answer now, using ONLY [N] bracket citations:"""

    result = ask_ollama(prompt)

    if not result["ok"]:
        print(f"\n\u274c {result['message']}")
        return None

    answer = result["response"]

    # Display answer
    print()
    print(bold('=' * 60))
    print(bold('\U0001f4ac  ANSWER'))
    print(bold('=' * 60))
    for line in answer.split('\n'):
        print(textwrap.fill(line, width=80, break_long_words=False, break_on_hyphens=False))

    # Extract ONLY citations that appear in the answer
    cited_keys = set()
    cited = []
    pattern = re.compile(r'\[(\d+)\]')

    for i, r in enumerate(top_results):
        key = f"[{i + 1}]"
        if pattern.search(answer):
            if key not in cited_keys:
                cited_keys.add(key)
                cited.append((i + 1, r))

    if cited:
        print('\n' + bold('-' * 60))
        print(bold('\U0001f4da  SOURCES CITED  ') + dim('(in order of citation)'))
        print(bold('-' * 60))

        # Create a mapping of citation numbers as they appear in the answer
        citation_order = {}
        for match in pattern.finditer(answer):
            cite_num = int(match.group(1))
            if cite_num not in citation_order:
                citation_order[cite_num] = len(citation_order) + 1

        # Sort cited sources by their appearance order in the answer
        cited.sort(key=lambda x: citation_order.get(x[0], 999))

        for orig_n, r in cited:
            d = r['hybrid_score']
            authority = r.get('authority', 'opinion')
            auth_badge = magenta('\u2696\ufe0f  REGULATORY') if authority == 'regulatory' \
                         else blue('\U0001f4ac  OPINION   ')
            if d < 0.5:
                rel_badge = green('\U0001f7e2 HIGH  ')
            elif d < 0.65:
                rel_badge = yellow('\U0001f7e1 MEDIUM')
            else:
                rel_badge = red('\U0001f534 LOW   ')
            fname = os.path.basename(r['source'])
            print(f'  {auth_badge}  {rel_badge}  {bold(f"[{orig_n}]")}  {bold(fname)}  --  page {r["page"]}')
            print(dim(f'         hybrid: {d:.3f}  |  rerank: {r["rerank_score"]:.2f}  |  semantic: {r["sem_score"]:.3f}  |  keyword: {r["kw_score"]:.3f}'))
            print()
    else:
        print('\n' + dim('\u26a0\ufe0f  No explicit citations found in the answer.'))
        print(dim('   Try switching to mistral or qwen2.5:14b in Block 6, or increase TOP_K above.'))

    return answer


# Run!
answer = ask_question(MY_QUESTION, top_k=TOP_K, final_k=FINAL_K)


## 💬 Block 8 — Interactive Q&A Loop

Ask multiple questions without re-running any cells.
Type your question and press Enter. Type `quit` or `exit` to stop.


In [ ]:
print("\U0001f9ed CDM Compass -- Interactive Q&A")
print("   Type your question and press Enter. Type 'quit' to stop.\n")

while True:
    question = input('\n\u2753 Your question: ').strip()

    if not question:
        print("   \u26a0\ufe0f  Please type a question.")
        continue

    if question.lower() in ("quit", "exit", "stop", "q"):
        print("\n\U0001f44b Goodbye!")
        break

    ask_question(question, top_k=TOP_K, final_k=FINAL_K)


---
## 📖 Quick Reference

### How CDM Compass finds answers
Each question goes through three stages:
1. **Hybrid search** — ChromaDB (meaning) + BM25 (keywords) retrieve up to `TOP_K` candidates
2. **Reranking** — a cross-encoder re-scores all candidates and keeps the best `FINAL_K`
3. **Compression** — only the most relevant sentences from each chunk reach the LLM

### Understanding source scores
Each cited source shows four scores:
- **hybrid** — combined semantic + keyword score (lower = better)
- **rerank** — cross-encoder relevance score (higher = better)
- **semantic** — meaning similarity to your question
- **keyword** — exact word overlap with your question

| Badge | Hybrid score | Meaning |
|-------|-------------|---------|
| 🟢 HIGH | < 0.5 | Strong match |
| 🟡 MEDIUM | 0.5 - 0.65 | Partial match |
| 🔴 LOW | > 0.65 | Weak match |

### Tuning
| Setting | Where | Effect |
|---------|-------|--------|
| `TOP_K` | Block 7 | More candidates before reranking — increase if answers miss known sources |
| `FINAL_K` | Block 7 | Chunks sent to LLM — increase for broader answers, decrease for precision |
| `SEMANTIC_WEIGHT` | Block 7 | Raise for conceptual questions |
| `KEYWORD_WEIGHT` | Block 7 | Raise for specific terms, IDs, clause numbers |
| `CHUNK_SIZE` | Block 4 | Lower (e.g. 200) for more precise citations |

### Tips
- Specific questions work better — *'What does ICH E6 R2 say about risk-based monitoring?'* beats *'tell me about RBQM'*
- `mistral` and `qwen2.5:14b` follow citation instructions most reliably
- If no sources appear, the LLM did not cite inline — increase `TOP_K` or rephrase the question

### Managing document versions
| Situation | What to do |
|-----------|------------|
| Updated document, old version no longer valid | Delete old file, drop in new one, re-run Blocks 3-5 |
| Keep both for comparison | Rename with version numbers (e.g. `v8` / `v9`), keep both in same subfolder, re-run Blocks 3-5 |
